In [14]:
# --- repo bootstrap: make src/ + this domain's config importable, run from repo root ---
# domains/credit_risk/ is mounted only in the credit-risk containers, so the
# domain-agnostic stages physically cannot import domain settings.
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
sys.path.insert(0, str(_ROOT / 'domains' / 'credit_risk'))
os.chdir(_ROOT)


# Aave V3.1 — model split (2h, by time, per-horizon embargo, leak-free)

Inputs: **`DF_model`** (features, one row per 2h `time_bucket`, already scaled) and
**`DF_target`** (forward targets `<base>_fwd_<h>`, one column per target and horizon). They stay
apart: this notebook only **defines** the split and the train-fitted steps, and writes them to
`model_data/batch_<span>/split/split_meta.json`. No joined data is written — training joins
one target at a time on `time_bucket` (`mds.join_for_training`) and keeps the rows
`msp.assign_split` labels for that target's horizon.

- **Batches**: `BATCH` is the same value as in `model_dataset` — one batch, or a set of batches
  joined there; the inputs are read from that run's `target_data/` / `model_data/` folder.
- **Holdout**: the last 15d — predicted, then validated when the next market data arrives.
- **70 / 15 / 15 by time** over the rest; boundaries are timestamps.
- **Embargo = the horizon**: a row is dropped when its label window `(t, t+h]` reaches the next
  split — 12 / 36 / 84 rows **per boundary** for 24h / 3d / 7d (twice that in total, one gap
  before val and one before test). **Warm-up**: the first 7d of train.
- **Blank features**: trailing 7d mean (past only) + a `_was_missing` flag, through
  `msp.prepare_features` — the same call training makes, so the features cannot drift.
- **Train-fit**: column selection (near-constant, mostly blank, |Spearman| duplicates — no
  reshaping) and mean/std for linear / KNN / SVM / MLP only; trees take the scores as they are.
- **Labels**: regression on every target; quantiles p10/p50/p90 (declared, trained later);
  binary stress for liquidation debt (train p75, per horizon).
- **CV**: 3 walk-forward folds inside train, embargo = horizon at each fold's train end.

In [15]:
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
from IPython.display import display

import transform as tf
import model_config as cfg
import model_dataset as mds
import model_split as msp

In [16]:
# --- split config ---
# the SAME value as model_dataset's BATCH: one batch ("2025-04") or a set joined there
BATCH = {"2024-04", "2025-04"}
# the two inputs: a CSV path, or (folder, table) for a frame saved with run.save
DF_MODEL_SRC  = ("model_data/batch_2024-04_2026-03/scaled_fixed/"
                 "DF_model_scaled_fixed_panel_2h_2024-04-01_2026-03-31.csv")
DF_TARGET_SRC = ("target_data", "DF_target_scaled_2h")
# DF_MODEL_SRC may also be a LIST of frames on one time grid, joined on time_bucket (msp.join_inputs), e.g.
# [the panel csv above, the same path with "_liq_" in place of "_panel_"]. Tested 2026-11-13 and NOT adopted:
# 24h ensemble CV 0.692 -> 0.705, val 0.562 -> 0.650, test 0.617 -> 0.588 (paired CI crosses 0 on test).

FRACTIONS = (0.70, 0.15, 0.15)  # train / val / test, by TIME, after the holdout
HOLDOUT   = "7d"               # last 7d: prediction + later validation, in no split
WARMUP    = "7d"                # first 7d of train: the trailing fill has no past there
N_FOLDS   = 3                   # walk-forward folds inside train

NON_FEATURES = ("time_bucket", "sampled_user_count", "asset_symbol")   # never model inputs
FILL_WINDOW  = "7d"
FILL_WEIGHT  = "sampled_user_count"
FILL_WEIGHTED = ("avg_total_debt_base", "risk_buffer", "overcollateral_margin",
                 "ltv_utilization")                 # per-bucket user averages: BASE names, rolled
                                                    # copies included; a missing name is printed
FILL = {"window": FILL_WINDOW, "weight": FILL_WEIGHT, "weighted": FILL_WEIGHTED}

MAX_NAN, MAX_CORR = 0.5, 0.95   # column selection, train rows only
PRIORITY = dict(cfg.CREDIT_RISK_WEIGHTS)            # by base name (rolled copies inherit, flags last)

STRESS_SPEC     = {"liquidation_debt_covered_value_usd": "high"}
STRESS_QUANTILE = 0.75
QUANTILES       = (0.1, 0.5, 0.9)                   # quantile models, trained later

In [ ]:
# --- load the two frames, each sorted by time; they meet only on time_bucket ---
# tf.use_output_batch only turns BATCH into the run's span (e.g. 2024-04 -> 2026-03); the data
# itself comes from DF_MODEL_SRC / DF_TARGET_SRC, resolved in that span's folders.
run = tf.use_output_batch(BATCH)
MODEL_DIR = run.folder(cfg.MODEL_DATA_DIR)
DF_model  = msp.load_input(DF_MODEL_SRC, batch=run.span)
DF_target = msp.load_input(DF_TARGET_SRC, batch=run.span)
# distinct key names from DF_MODEL_SRC/DF_TARGET_SRC on purpose — the export cell below merges
# this dict in, and an equal key ("DF_model") would silently overwrite the real source with
# this fingerprint instead of just adding to it
INPUTS = {"DF_model_fingerprint": msp.frame_fingerprint(DF_model),
         "DF_target_fingerprint": msp.frame_fingerprint(DF_target)}
mds.assert_separate(DF_model, DF_target)            # no target column (base, rolled, forward) in the features

TARGET_COLS = [c for c in DF_target.columns if "_fwd_" in c]
HORIZONS = list(dict.fromkeys(c.rpartition("_fwd_")[2] for c in TARGET_COLS))   # e.g. ['24h', '3d', '7d']
model_t, target_t = msp.time_of(DF_model), msp.time_of(DF_target)
assert set(model_t) == set(target_t), \
    f"grids differ: {len(set(model_t) - set(target_t))} stamp(s) only in DF_model, " \
    f"{len(set(target_t) - set(model_t))} only in DF_target"
print(f" {run}")
print(f" DF_model {DF_model.shape}, DF_target {DF_target.shape}: {len(TARGET_COLS)} targets, "
      f"horizons {HORIZONS}")

In [18]:
# --- blank features: trailing 7d mean (past only) + a _was_missing flag ---
# msp.prepare_features is the ONE entry point: training calls it with the saved meta (which then
# also applies the column selection), so the fill here and there is the same code.
DF_model = msp.prepare_features(DF_model, {"fill": FILL, "non_features": NON_FEATURES})
FEATURE_COLS = msp.feature_columns(DF_model, NON_FEATURES)
print(f" {len(FEATURE_COLS)} feature columns (flags included)")

 ⚠️ weighted-fill base(s) not in DF_model, filled unweighted if present: ['avg_total_debt_base', 'risk_buffer', 'overcollateral_margin', 'ltv_utilization']
 trailing 7d means added for 3 column(s)
 trailing 7d fill: 3 column(s) had blanks, 109 cell(s) filled, 2 still blank, 3 '_was_missing' flag(s) added
 48 feature columns (flags included)


In [19]:
# --- boundaries by time: holdout = last 15d, then 70/15/15 of the rest ---
BOUNDS = msp.split_boundaries(target_t, FRACTIONS, holdout=HOLDOUT)
display(msp.boundary_table(BOUNDS, target_t))

,split,first,last,rows,days
0,train,2024-04-01 02:00:00+00:00,2025-08-19 10:00:00+00:00,6053,505.3
1,val,2025-08-19 12:00:00+00:00,2025-12-05 16:00:00+00:00,1299,108.2
2,test,2025-12-05 18:00:00+00:00,2026-03-24 00:00:00+00:00,1300,108.2
3,holdout,2026-03-24 02:00:00+00:00,2026-03-31 00:00:00+00:00,84,6.9


In [20]:
# --- row labels per horizon: train / val / test, or embargo / warmup / holdout ---
ASSIGN = {h: msp.assign_split(DF_target, BOUNDS, h, warmup=WARMUP) for h in HORIZONS}
display(msp.assignment_table(ASSIGN))
display(msp.split_summary(DF_target, ASSIGN))   # labelled rows per target per split; pending = to predict

,train,val,test,embargo,warmup,holdout
24h,5957,1287,1300,24,84,84
3d,5933,1263,1300,72,84,84
7d,5885,1215,1300,168,84,84


,column,train,val,test,holdout,pending
0,liquidation_debt_covered_value_usd_fwd_24h,5945,1287,1300,72,12
1,liquidation_debt_covered_value_usd_fwd_3d,5897,1263,1300,48,36
2,liquidation_debt_covered_value_usd_fwd_7d,5801,1215,1300,0,84
3,net_liquidity_flow_usd_fwd_24h,5945,1287,1300,72,12
4,net_liquidity_flow_usd_fwd_3d,5897,1263,1300,48,36
5,net_liquidity_flow_usd_fwd_7d,5801,1215,1300,0,84
6,avg_ltv_fwd_24h,5828,1273,1290,72,12
7,avg_ltv_fwd_3d,5897,1263,1300,48,36
8,avg_ltv_fwd_7d,5801,1215,1300,0,84
9,borrow_capacity_utilization_fwd_24h,5821,1270,1290,72,12


In [21]:
# --- train-fit column selection, on the strictest horizon's train rows (no reshaping) ---
TRAIN_STAMPS = set(target_t[ASSIGN[HORIZONS[-1]] == "train"])
train_X = DF_model[model_t.isin(TRAIN_STAMPS)]
SELECTION = msp.fit_column_selection(train_X, FEATURE_COLS, max_nan=MAX_NAN, max_corr=MAX_CORR,
                                     priority=PRIORITY)
KEPT = SELECTION["kept"]
display(pd.DataFrame(SELECTION["reasons"].items(), columns=["dropped", "reason"]))

 column selection on 5885 train rows: kept 43 of 48


,dropped,reason
0,borrow_growth_was_missing,near-constant (100.0% one value)
1,liquidity_growth_rate_was_missing,near-constant (100.0% one value)
2,supply_tx_count_rolled_12h,|spearman| 0.978 with collateral_enabled_count...
3,supply_tx_count_rolled_3d,|spearman| 0.987 with collateral_enabled_count...
4,supply_tx_count_rolled_7d,|spearman| 0.952 with collateral_enabled_count...


In [22]:
# --- standardizer: train mean / std (median for any blank left) — linear, KNN, SVM, MLP only.
# Trees take the scores as they are; targets are never standardized (they reverse-scale).
SCALER = msp.fit_scaler(train_X, KEPT)
display(pd.DataFrame(SCALER).T.head())

,mean,std,median
borrow_amount_value_usd,64.797691,5.042121,64.790479
unique_borrowers,40.165258,5.768921,40.051065
repay_amount_value_usd,64.300432,5.235903,64.422397
net_borrow_demand_usd,51.764306,14.050561,52.826280
borrow_repay_ratio,62.073539,42.080045,52.954050


In [23]:
# --- binary stress: train-only threshold per horizon (each on its own horizon's train rows) ---
STRESS = msp.stress_thresholds(DF_target, STRESS_SPEC, ASSIGN, quantile=STRESS_QUANTILE)
display(pd.DataFrame(STRESS).T)
DF_stress = msp.binarize_stress(DF_target, STRESS)   # built here only to check balance; not saved
display(msp.stress_balance(DF_stress, ASSIGN))

,side,quantile,threshold
liquidation_debt_covered_value_usd_fwd_24h,high,0.75,46.067964
liquidation_debt_covered_value_usd_fwd_3d,high,0.75,55.979115
liquidation_debt_covered_value_usd_fwd_7d,high,0.75,61.617179


,label,train,val,test
0,y_stress_liquidation_debt_covered_value_usd_24h,0.250,0.385,0.356
1,y_stress_liquidation_debt_covered_value_usd_3d,0.250,0.447,0.272
2,y_stress_liquidation_debt_covered_value_usd_7d,0.249,0.462,0.346


In [24]:
# --- leakage guards per horizon + one join preview (the only place features meet a target) ---
for h in HORIZONS:
    msp.assert_no_leakage(DF_target, ASSIGN[h], BOUNDS, h)
preview_col = TARGET_COLS[0]
preview_h = preview_col.rpartition("_fwd_")[2]     # its own horizon's labels, not the longest one's
joined = mds.join_for_training(DF_model[["time_bucket", *KEPT]], DF_target, preview_col)
train_rows = joined[msp.time_of(joined).isin(set(target_t[ASSIGN[preview_h] == "train"]))]
print(f" {preview_col}: {train_rows.shape[0]} train rows x {train_rows.shape[1]} cols after the join")

 no-leakage OK  24h: train < val < test, every label window inside its split
 no-leakage OK   3d: train < val < test, every label window inside its split
 no-leakage OK   7d: train < val < test, every label window inside its split
 joined liquidation_debt_covered_value_usd_fwd_24h: 8712 labelled row(s) of 8736
 liquidation_debt_covered_value_usd_fwd_24h: 5945 train rows x 45 cols after the join


In [25]:
# --- walk-forward folds inside train, per horizon (embargo = horizon at each fold's end) ---
FOLDS = {}
for h in HORIZONS:
    is_train = (ASSIGN[h] == "train").to_numpy()
    stamps = target_t[is_train].reset_index(drop=True)
    folds = msp.walk_forward_folds(stamps, n_splits=N_FOLDS, horizon=h)
    y_col = next((msp.stress_name(c) for c in STRESS if c.endswith(f"_fwd_{h}")), None)
    y = DF_stress.loc[is_train, y_col].to_numpy() if y_col else None
    FOLDS[h] = msp.fold_summary(stamps, folds, y)
    print(f"== {h}" + (f" (positives: {y_col})" if y_col else ""))
    display(FOLDS[h])

== 24h (positives: y_stress_liquidation_debt_covered_value_usd_24h)


,fold,train_rows,train_last,test_first,test_last,test_rows,test_labelled,test_positives
0,1,1480,2024-08-09 08:00:00+00:00,2024-08-10 10:00:00+00:00,2024-12-12 16:00:00+00:00,1492,1492,292
1,2,2972,2024-12-11 16:00:00+00:00,2024-12-12 18:00:00+00:00,2025-04-16 00:00:00+00:00,1480,1468,459
2,3,4452,2025-04-15 00:00:00+00:00,2025-04-16 02:00:00+00:00,2025-08-18 10:00:00+00:00,1493,1493,408


== 3d (positives: y_stress_liquidation_debt_covered_value_usd_3d)


,fold,train_rows,train_last,test_first,test_last,test_rows,test_labelled,test_positives
0,1,1450,2024-08-06 20:00:00+00:00,2024-08-09 22:00:00+00:00,2024-12-11 16:00:00+00:00,1486,1486,251
1,2,2936,2024-12-08 16:00:00+00:00,2024-12-11 18:00:00+00:00,2025-04-14 12:00:00+00:00,1474,1438,593
2,3,4410,2025-04-11 12:00:00+00:00,2025-04-14 14:00:00+00:00,2025-08-16 10:00:00+00:00,1487,1487,228


== 7d (positives: y_stress_liquidation_debt_covered_value_usd_7d)


,fold,train_rows,train_last,test_first,test_last,test_rows,test_labelled,test_positives
0,1,1390,2024-08-01 20:00:00+00:00,2024-08-08 22:00:00+00:00,2024-12-09 16:00:00+00:00,1474,1474,211
1,2,2864,2024-12-02 16:00:00+00:00,2024-12-09 18:00:00+00:00,2025-04-11 12:00:00+00:00,1462,1378,721
2,3,4326,2025-04-04 12:00:00+00:00,2025-04-11 14:00:00+00:00,2025-08-12 10:00:00+00:00,1475,1475,106


In [26]:
# --- export the split DEFINITION only -> model_data/batch_<span>/split/split_meta.json ---
meta = {"inputs": {"DF_model": DF_MODEL_SRC, "DF_target": DF_TARGET_SRC, "batch": list(run.span),
                   **INPUTS},                      # rows / cols / first / last: training checks them
        "boundaries": BOUNDS, "fractions": FRACTIONS, "holdout": HOLDOUT, "warmup": WARMUP,
        "embargo": {h: h for h in HORIZONS},           # embargo = the horizon itself
        "fill": FILL,                                  # replayed by msp.prepare_features(DF_model, meta)
        "non_features": NON_FEATURES, "feature_cols": FEATURE_COLS,
        "selection": SELECTION, "scaler_params": SCALER,
        "standardize_for": ["linear", "knn", "svm", "mlp"],
        "targets": TARGET_COLS, "quantiles": QUANTILES,
        "stress": {"spec": STRESS_SPEC, "quantile": STRESS_QUANTILE, "thresholds": STRESS},
        "folds": {"n": N_FOLDS, **FOLDS}}
msp.save_split_meta(meta, os.path.join(MODEL_DIR, "split"))

 wrote split definition -> model_data/batch_2024-04_2026-03/split/split_meta.json


'model_data/batch_2024-04_2026-03/split/split_meta.json'